# Day 03 · The data stack: NumPy, Pandas, Matplotlib (+ Git)

**Goal:** delete the loops. Compute with **NumPy** (vectorisation, broadcasting), wrangle with **Pandas** (load → inspect → clean → filter → groupby → merge), look with **Matplotlib/Seaborn**, and push the result to **GitHub**. Dataset: the Titanic (891 passengers) — the "hello world" of data science.

**Run in Google Colab:** File → Open notebook → Upload → pick this `.ipynb`. Then *Runtime → Run all*.

**Runtime:** CPU is fine.

| Section | What you'll do |
|---|---|
| 1 | NumPy: arrays, shapes, vectorisation (with a timing race) |
| 2 | Broadcasting, indexing, views vs copies |
| 3 | Pandas: load Titanic, the four first looks, missing values |
| 4 | Filter, sort, new columns, `apply` |
| 5 | groupby / pivot / merge — split-apply-combine |
| 6 | Matplotlib & Seaborn: the plots for your README |
| 7 | Git & GitHub: push this notebook |
| 8 | 🎯 Try it yourself + Homework starter |

## 0. Setup
All four libraries are preinstalled in Colab. We print versions (pandas ≥ 2.x is what we assume) and fix the random seed.

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import time, os
np.random.seed(42)
sns.set_theme(style="whitegrid")
print("numpy", np.__version__, "| pandas", pd.__version__, "| matplotlib", __import__("matplotlib").__version__, "| seaborn", sns.__version__)

numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | seaborn 0.13.2


## 1. NumPy: think in shapes, not loops 🍬

**Analogy:** a box of 12 laddoos in 3 rows of 4 is an array of shape `(3, 4)`. "Add 5 g sugar to every laddoo" is `box + 5` — one instruction, no loop. That is **vectorisation**: the loop still happens, but in compiled C over a contiguous block of one dtype, not in Python over boxed objects.

Every model input you will ever see is an n-dimensional array: a grayscale digit `(28, 28)`, a colour photo `(H, W, 3)`, a batch `(64, 224, 224, 3)`, a tokenised sentence `(seq_len,)`.

In [2]:
a = np.array([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]])
print(a)
print("shape:", a.shape, "| ndim:", a.ndim, "| dtype:", a.dtype, "| size:", a.size)
print("a + 5        ->\n", a + 5)                 # scalar broadcast
print("a * a        ->\n", a * a)                 # element-wise (NOT matrix multiply)
print("a.sum()      ->", a.sum())
print("a.sum(axis=0)->", a.sum(axis=0), "  <- down the rows: one number per column")
print("a.sum(axis=1)->", a.sum(axis=1), "     <- across the columns: one number per row")
print("a.mean(), a.max(), a.argmax():", a.mean(), a.max(), a.argmax())

[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]
shape: (3, 4) | ndim: 2 | dtype: int64 | size: 12
a + 5        ->
 [[ 6  7  8  9]
 [10 11 12 13]
 [14 15 16 17]]
a * a        ->
 [[  1   4   9  16]
 [ 25  36  49  64]
 [ 81 100 121 144]]
a.sum()      -> 78
a.sum(axis=0)-> [15 18 21 24]   <- down the rows: one number per column
a.sum(axis=1)-> [10 26 42]      <- across the columns: one number per row
a.mean(), a.max(), a.argmax(): 6.5 12 11


In [3]:
# Handy constructors you'll use constantly
print(np.zeros((2, 3)))
print(np.ones(4, dtype=int))
print(np.arange(0, 10, 2))                 # like range
print(np.linspace(0, 1, 5))                # 5 evenly spaced numbers, inclusive
rng = np.random.default_rng(0)             # the modern random API
print(rng.normal(loc=0, scale=1, size=(2, 3)).round(2))
print(a.reshape(4, 3).shape, a.reshape(-1).shape, a.T.shape)   # -1 = "work it out"

[[0. 0. 0.]
 [0. 0. 0.]]
[1 1 1 1]
[0 2 4 6 8]
[0.   0.25 0.5  0.75 1.  ]
[[ 0.13 -0.13  0.64]
 [ 0.1  -0.54  0.36]]
(4, 3) (12,) (4, 3)


### The race: Python loop vs NumPy
Same job — sum of squares of a million numbers. Watch the ratio.

In [6]:
N = 1_000_000
xs = list(range(N))
arr = np.arange(N)                       # int64, so the sums are exact and comparable
_ = np.square(arr).sum(); _ = arr @ arr  # warm-up (first call pays a one-time setup cost)

t0 = time.perf_counter(); s1 = sum(x * x for x in xs);  t_py  = time.perf_counter() - t0
t0 = time.perf_counter(); s2 = np.square(arr).sum();     t_np  = time.perf_counter() - t0
t0 = time.perf_counter(); s3 = arr @ arr;                t_dot = time.perf_counter() - t0   # dot product = same maths, one BLAS call

print(f"python loop      : {t_py*1000:7.2f} ms")
print(f"np.square().sum(): {t_np*1000:7.2f} ms   ({t_py/t_np:.0f}x faster)")
print(f"numpy a @ a      : {t_dot*1000:7.2f} ms   ({t_py/t_dot:.0f}x faster)")
print("same answer:", s1 == s2 == s3)

python loop      :  162.50 ms
np.square().sum():    2.64 ms   (62x faster)
numpy a @ a      :    1.93 ms   (84x faster)
same answer: True


## 2. Broadcasting, indexing, views vs copies 📐

**Broadcasting rule:** align shapes from the **right**; each pair of dimensions must be equal or one of them must be 1; the 1 is stretched (virtually — no memory copied). Missing leading dims count as 1.

- `(3,4) + ()` ✅ scalar → every cell
- `(3,4) + (4,)` ✅ one row → added to every row
- `(3,4) + (3,1)` ✅ one column → added to every column
- `(3,4) + (3,)` ❌ 4 vs 3, neither is 1 → `ValueError`

In [7]:
A = np.arange(1, 13).reshape(3, 4)
row = np.array([10, 20, 30, 40])          # (4,)
col = np.array([100, 200, 300])           # (3,)

print("A + row (4,) ->\n", A + row)
print("A + col[:, None] (3,1) ->\n", A + col[:, None])      # [:, None] adds an axis -> column vector
try:
    A + col                                                # (3,4) + (3,) -> error
except ValueError as e:
    print("ValueError:", e)

# The real use: standardise every column of a feature matrix in ONE line
X = rng.normal(size=(891, 4)) * [1, 10, 100, 1000] + [5, 50, 500, 5000]   # 4 features on wildly different scales
X_std = (X - X.mean(axis=0)) / X.std(axis=0)                # (891,4) - (4,) broadcasts
print("column means after standardising:", X_std.mean(axis=0).round(6))
print("column stds  after standardising:", X_std.std(axis=0).round(6))

A + row (4,) ->
 [[11 22 33 44]
 [15 26 37 48]
 [19 30 41 52]]
A + col[:, None] (3,1) ->
 [[101 102 103 104]
 [205 206 207 208]
 [309 310 311 312]]
ValueError: operands could not be broadcast together with shapes (3,4) (3,) 
column means after standardising: [-0.  0. -0. -0.]
column stds  after standardising: [1. 1. 1. 1.]


In [ ]:
# Four kinds of indexing — and which ones give a VIEW (shared memory) vs a COPY
a = np.arange(12).reshape(3, 4)
print(a)
print("a[1, 2]      =", a[1, 2])              # position
print("a[:, 0]      =", a[:, 0])              # first column (a VIEW)
print("a[a > 6]     =", a[a > 6])             # boolean mask (a COPY)
print("a[[0, 2]]    =\n", a[[0, 2]])          # fancy indexing (a COPY)

v = a[:, 0]          # view
v[0] = 999           # ...so writing into it changes a!
print("after v[0]=999, a[0,0] =", a[0, 0], "  <- Day 01's sticker problem, at the memory level")

c = a[a > 6]         # copy
c[0] = -1
print("after c[0]=-1, a still has no -1:", (a == -1).any())
print("safe habit when unsure: b = a[:, 0].copy()")

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
a[1, 2]      = 6
a[:, 0]      = [ 2  6 10]
a[a > 6]     = [ 7  8  9 10 11]
a[[0, 2]]    =
 [[ 0  1  2  3]
 [ 8  9 10 11]]
after v[0]=999, a[0,0] = 999   <- Day 01's sticker problem, at the memory level
after c[0]=-1, a still has no -1: False
safe habit when unsure: b = a[:, 0].copy()


> 🧠 **Interview Q:** *"Why NumPy instead of Python lists?"*
> Contiguous block of one dtype → operations run as compiled loops over raw memory (10–100× faster, far less RAM) and broadcasting lets me write whole-array expressions like `(X - X.mean(0)) / X.std(0)` instead of nested loops.

## 3. Pandas: load the Titanic and take the four first looks 🚢

A **DataFrame** is labelled columns (each a NumPy array underneath) with an index. The four things you *always* do first: `shape`, `dtypes`/`info()`, `head()`, `describe()` — then `isna().sum()`.

In [10]:
URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
try:
    df = pd.read_csv(URL)
    print("loaded from GitHub")
except Exception as e:                                   # offline fallback so the notebook still runs
    print("could not download (", type(e).__name__, ") -> trying seaborn's cached copy")
    df = sns.load_dataset("titanic")

print("shape:", df.shape)                                 # (891, 15)
df.head()

loaded from GitHub
shape: (891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    object 
 3   age          714 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         891 non-null    float64
 7   embarked     889 non-null    object 
 8   class        891 non-null    object 
 9   who          891 non-null    object 
 10  adult_male   891 non-null    bool   
 11  deck         203 non-null    object 
 12  embark_town  889 non-null    object 
 13  alive        891 non-null    object 
 14  alone        891 non-null    bool   
dtypes: bool(2), float64(2), int64(4), object(7)
memory usage: 92.4+ KB


In [12]:
df.describe().round(2)          # numeric columns only; add include="all" to see categoricals

,survived,pclass,age,sibsp,parch,fare
count,891.00,891.00,714.00,891.00,891.00,891.00
mean,0.38,2.31,29.70,0.52,0.38,32.20
std,0.49,0.84,14.53,1.10,0.81,49.69
min,0.00,1.00,0.42,0.00,0.00,0.00
25%,0.00,2.00,20.12,0.00,0.00,7.91
50%,0.00,3.00,28.00,0.00,0.00,14.45
75%,1.00,3.00,38.00,1.00,0.00,31.00
max,1.00,3.00,80.00,8.00,6.00,512.33


In [13]:
missing = df.isna().sum()
print(missing[missing > 0])
print(f"\ndeck is {df['deck'].isna().mean():.0%} missing -> drop it")
print(f"age  is {df['age'].isna().mean():.0%} missing -> fill it (per class median)")

age            177
embarked         2
deck           688
embark_town      2
dtype: int64

deck is 77% missing -> drop it
age  is 20% missing -> fill it (per class median)


### Missing values: decide per column, don't ignore
- **Drop** when mostly empty (`deck`: 77%).
- **Fill** with a robust statistic when a minority is missing (`age`: median *per passenger class* — first-class passengers were older).
- **Flag** when missingness itself may be a signal (`age_missing`).

Never fill with 0 silently — a 0-year-old is a real value.

In [14]:
df["age_missing"] = df["age"].isna()                                          # flag first
df["age"] = df["age"].fillna(df.groupby("pclass")["age"].transform("median")) # fill per class
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df["embark_town"] = df["embark_town"].fillna(df["embark_town"].mode()[0])
df = df.drop(columns=["deck"])

assert df.isna().sum().sum() == 0, "still has NaNs!"
print("clean ✅  shape:", df.shape)
print(df.groupby("pclass")["age"].median())

clean ✅  shape: (891, 15)
pclass
1    37.0
2    29.0
3    24.0
Name: age, dtype: float64


## 4. Filter, sort, new columns, `apply` 🔧

Whole-column arithmetic reads like English: *fare per head = fare / family size*. Filtering uses boolean masks exactly like NumPy. `loc` = by **label**, `iloc` = by **position**.

In [15]:
df["family"] = df["sibsp"] + df["parch"] + 1               # new column from old ones
df["fare_per_head"] = df["fare"] / df["family"]

rich_kids = df[(df["age"] < 18) & (df["class"] == "First")]   # & | ~ for and/or/not; parentheses required
print("first-class children:", len(rich_kids))
print(df.query("age < 18 and pclass == 1").shape[0], "<- same thing with .query")

print(df.loc[df["fare"] > 500, ["who", "fare", "embark_town", "survived"]])   # loc: labels + mask
print(df.iloc[0:3, 0:4])                                                       # iloc: positions

print(df.sort_values(["class", "fare"], ascending=[True, False]).head(3)[["class", "fare", "who"]])

first-class children: 12
12 <- same thing with .query
       who      fare embark_town  survived
258  woman  512.3292   Cherbourg         1
679    man  512.3292   Cherbourg         1
737    man  512.3292   Cherbourg         1
   survived  pclass     sex   age
0         0       3    male  22.0
1         1       1  female  38.0
2         1       3  female  26.0
     class      fare    who
258  First  512.3292  woman
679  First  512.3292    man
737  First  512.3292    man


In [ ]:
# apply: run a Python function per row/element. Use it when there is no vectorised way; it is slower.
def age_group(age):
    if age < 13:  return "child"
    if age < 60:  return "adult"
    return "senior"

df["age_group"] = df["age"].apply(age_group)
print(df["age_group"].value_counts())

# The vectorised way (preferred for speed): pd.cut
df["age_bin"] = pd.cut(df["age"], bins=[0, 12, 59, 120], labels=["child", "adult", "senior"])
print((df["age_bin"].astype(str) == df["age_group"]).all(), "<- identical result, ~100x faster on big data")

## 5. groupby, pivot, merge — split · apply · combine 🏏

**Analogy:** the IPL points table. One long log of matches → **split** by team → **apply** "count wins" to each pile → **combine** into one row per team. `df.groupby(key)[col].agg(...)` is that move. Every dashboard is this.

In [ ]:
# Q1: survival rate by class and sex  (the whole story of "women and children first")
rate = df.groupby(["class", "sex"], observed=True)["survived"].mean()
print(rate.round(3))

# Q2: a 2-D table of the same thing
print(pd.pivot_table(df, values="survived", index="class", columns="sex", aggfunc="mean", observed=True).round(3))

# Q3: several aggregations at once, with named output columns
summary = df.groupby("class", observed=True).agg(
    n=("survived", "size"),
    survival_rate=("survived", "mean"),
    median_fare=("fare", "median"),
    mean_age=("age", "mean"),
).round(2)
summary

In [ ]:
# merge = SQL JOIN. Attach a small lookup table of port names/countries to every passenger.
ports = pd.DataFrame({"embarked": ["S", "C", "Q"],
                      "port": ["Southampton", "Cherbourg", "Queenstown"],
                      "country": ["England", "France", "Ireland"]})
before = len(df)
merged = pd.merge(df, ports, on="embarked", how="left")     # left: keep every passenger
print("rows before/after merge:", before, len(merged), "<- ALWAYS check; a many-to-many join multiplies rows")
print(merged.groupby("country")["survived"].agg(["size", "mean"]).round(3))

> 🧠 **Interview Q:** *"Left join vs inner join?"* Left keeps every row of the left table and fills NaN where no match; inner keeps only rows that match in both. Outer keeps everything from both.

## 6. Matplotlib & Seaborn: the plots for your README 📊

Two lines to memorise: `fig, ax = plt.subplots()` then `ax.plot(...)`/`sns.xxxplot(..., ax=ax)`. Always label axes and give a title — an unlabelled chart in a portfolio is a red flag.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ORDER = ["First", "Second", "Third"]
sns.barplot(df, x="class", y="survived", hue="sex", order=ORDER, ax=axes[0])
axes[0].set(title="Survival rate by class & sex", ylabel="P(survived)", ylim=(0, 1))

sns.histplot(df, x="age", hue="survived", bins=30, element="step", ax=axes[1])
axes[1].set(title="Age distribution by outcome")   # the spike at 24 is OUR median fill for 3rd class -> why we kept the age_missing flag

sns.boxplot(df, x="class", y="fare", order=ORDER, ax=axes[2])
axes[2].set(title="Fare by class", yscale="log")

fig.tight_layout()
fig.savefig("titanic_eda.png", dpi=150, bbox_inches="tight")   # <- this PNG goes into your README
print("saved titanic_eda.png:", os.path.getsize("titanic_eda.png") // 1024, "KB")
plt.show()

In [ ]:
# Pure matplotlib, for when you need control: a line plot of survival rate vs age bin
bins = pd.cut(df["age"], bins=range(0, 85, 5))
curve = df.groupby(bins, observed=True)["survived"].mean()

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot([b.left for b in curve.index], curve.values, marker="o", color="tab:orange")
ax.axhline(df["survived"].mean(), ls="--", color="gray", label=f"overall {df['survived'].mean():.2f}")
ax.set(title="Survival rate by 5-year age band", xlabel="age (lower edge of band)", ylabel="P(survived)")
ax.legend(); ax.grid(alpha=.3)
plt.show()

# Pandas' one-liner shortcut for quick looks:
df["fare"].plot(kind="hist", bins=40, title="fare (pandas .plot)"); plt.show()

In [ ]:
# A correlation heatmap of the numeric columns — the classic EDA picture
num = df.select_dtypes("number").drop(columns=["pclass"])
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(num.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation between numeric columns")
plt.show()
df.to_csv("titanic_clean.csv", index=False)
print("saved titanic_clean.csv")

## 7. Git & GitHub: push this notebook 🚀

Three areas, four commands: **working directory** → `git add` → **staging** → `git commit` → **local repo** → `git push` → **GitHub**.

**Easiest path from Colab (do this today):** `File → Save a copy in GitHub` → choose your `ai-ml-45days` repo → commit message `Day 03: Titanic EDA` → OK. Done — no terminal.

**The terminal path (learn it this week):** the cell below runs the same commands in a throw-away local repo so you can see the messages Git prints. It does *not* push (that needs your credentials); the push command is shown at the end.

In [ ]:
%%bash
rm -rf /tmp/demo_repo && mkdir -p /tmp/demo_repo && cd /tmp/demo_repo
git init -q -b main
git config user.email "you@example.com" && git config user.name "Your Name"
echo "# Day 03 — Titanic EDA" > README.md
cp /content/titanic_eda.png . 2>/dev/null || cp "$OLDPWD/titanic_eda.png" . 2>/dev/null || echo "(png not found here, fine)"
echo "--- git status BEFORE add ---"; git status --short
git add README.md titanic_eda.png 2>/dev/null || git add README.md
echo "--- git status AFTER add (staged = green A) ---"; git status --short
git commit -q -m "Day 03: Titanic EDA with pandas + seaborn"
echo "--- git log ---"; git log --oneline
echo
echo "Next, on your laptop (once): git remote add origin https://github.com/<you>/ai-ml-45days.git"
echo "Then every day:            git push -u origin main"

> 💼 **Career note:** a recruiter searching "pandas" on LinkedIn finds you *only* if the word is in your Skills section. Add **Pandas, NumPy, Data Analysis, Git** today, and make the repo README show `titanic_eda.png` with one sentence of insight: *"First-class women survived at 97%, third-class men at 14%."*

## 8. 🎯 Try it yourself (in class, 5 minutes)

Using **groupby only** (no loops):
1. Survival rate by `embark_town`.
2. Median `fare` by `class` **and** `alone`.
3. Which `who` group (man / woman / child) had the highest survival rate, and what was it?

In [ ]:
# YOUR CODE HERE
# q1 = df.groupby(...)["survived"].mean()
# q2 = df.groupby([...], observed=True)["fare"].median()
# q3 = df.groupby("who")["survived"].mean().idxmax(), ...

<details><summary>Solution</summary>

```python
q1 = df.groupby("embark_town")["survived"].mean().round(3)
q2 = df.groupby(["class", "alone"], observed=True)["fare"].median()
who_rate = df.groupby("who")["survived"].mean()
q3 = (who_rate.idxmax(), round(who_rate.max(), 3))
print(q1, q2, q3, sep="\n\n")
# embark_town: Cherbourg 0.554, Queenstown 0.390, Southampton 0.339
# q3: ('woman', 0.756)
```
</details>

## Homework starter 🏠

- **Easy:** the three questions above + one saved, labelled plot (`sns.histplot(hue=...)`).
- **Medium (daily life):** load *your own* CSV (Swiggy/Zomato orders, UPI statement, screen time — or type 30 rows). Spend per weekday, top-5 merchants, monthly line plot. Commit (strip personal identifiers first).
- **Stretch:** Kaggle *Indian Startup Funding* — clean `"1,50,00,000"`-style amounts and inconsistent city names, then plot funding by year and top-10 cities.

Scaffold for the Medium task (generates a fake order history so the cell runs; replace with your real file):

In [ ]:
# --- Medium: your own spending data ---
rng = np.random.default_rng(7)
n = 120
orders = pd.DataFrame({
    "date": pd.to_datetime("2026-01-01") + pd.to_timedelta(rng.integers(0, 240, n), unit="D"),
    "merchant": rng.choice(["Swiggy", "Zomato", "Blinkit", "Chai Point", "IRCTC", "Amazon"], n, p=[.3,.25,.15,.15,.05,.1]),
    "amount": rng.lognormal(5.2, 0.6, n).round(0),        # replace with: pd.read_csv("my_orders.csv")
})
orders["weekday"] = orders["date"].dt.day_name()
orders["month"] = orders["date"].dt.to_period("M")

print("spend per weekday:\n", orders.groupby("weekday")["amount"].sum().sort_values(ascending=False).round(0))
print("\ntop merchants:\n", orders.groupby("merchant")["amount"].agg(["size", "sum"]).sort_values("sum", ascending=False).head())

monthly = orders.groupby("month")["amount"].sum()
ax = monthly.plot(kind="line", marker="o", title="Monthly spend (₹)", figsize=(8, 3.5))
ax.set(xlabel="month", ylabel="₹"); plt.show()
# YOUR CODE HERE: replace the fake data with your CSV, then commit the notebook.

## Key takeaways 🔑

- **NumPy:** one dtype, one shape, whole-array expressions. `axis=0` goes down the rows (one result per column). Slices are **views**, masks are **copies**.
- **Broadcasting:** align from the right; equal or 1. `(891,4) − (4,)` standardises columns in one line; `b[:, None]` turns a row into a column.
- **Pandas first looks:** `shape`, `info()`, `head()`, `describe()`, `isna().sum()`. Then per column: drop / fill (per-group median) / flag.
- **groupby = split-apply-combine.** `agg(name=(col, func))` for several stats; `pivot_table` for 2-D; `merge(how="left")` and check the row count.
- **Plots:** `fig, ax = plt.subplots()`, Seaborn for exploration, always label, `fig.savefig("x.png", dpi=150)` for the README.
- **Git:** add → commit → push. From Colab: *File → Save a copy in GitHub*.

**Before Day 04:** Days 01–03 notebooks + `titanic_eda.png` visible on GitHub; watch 3Blue1Brown's *Gradient descent* video — tomorrow the machine starts learning.